# 01 · Data check
Run this after every pull, before any modelling. It answers: how much data, which dates and patches, and whether anything looks broken.
Pull first from the `ml/` folder: `python -m broadway_ml.pull probe`, then `python -m broadway_ml.pull matches --days 14` (or `crawl`).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}

In [ ]:
m, p = T['matches'], T['players']
print('matches', len(m), '| from', pd.to_datetime(m.start_time.min(), unit='s'), 'to', pd.to_datetime(m.start_time.max(), unit='s'))
print('team 0 (Hidden King) win rate:', round((m.winning_team == 0).mean(), 4))
print('players per match:', p.groupby('match_id').size().value_counts().to_dict())
print('heroes seen:', p.hero_id.nunique())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
m.average_badge.plot.hist(bins=40, ax=ax[0], title='average badge')
(m.duration_s / 60).plot.hist(bins=40, ax=ax[1], title='duration (min)')
pd.to_datetime(m.start_time, unit='s').dt.date.value_counts().sort_index().plot(ax=ax[2], title='matches per day')

**Patches.** A balance patch changes hero strength, so the draft model should train on games from the current patch (or carry a patch feature). Mark the patch dates from `/v2/patches` here and decide the training window.

In [ ]:
# Patch dates, e.g. from the site's api-samples/patches.json or a fresh call
PATCH_CUTOFF = None  # unix time of the last balance patch; filter m = m[m.start_time >= PATCH_CUTOFF]